# Lab 5 - Text Representation

Completed solutions to all five tasks: TF-IDF cosine similarity, TF-IDF word importance, Skip-gram Word2Vec training, nearest words, and odd-one-out queries.

## Inputs and assignment check

- **Tasks 1 and 2:** use the exact sentences specified in the handout.
- **Tasks 3 to 5:** the handout explicitly requires `simpsons_script_lines.csv`, the `spoken_words` column, and Simpsons character names. This is the assigned dataset, not an interchangeable example.
- The fake/real news dataset belongs to the earlier teaching demonstration and is not required for these tasks.

The handout did not include a Simpsons download link or the CSV. The required file is now available locally from [The Simpsons Dataset on Kaggle](https://www.kaggle.com/datasets/prashant111/the-simpsons-dataset). Run the repository setup and download commands before rerunning after a fresh clone. No additional input is needed to complete the listed tasks.

TF-IDF is a sparse term-weight representation; Word2Vec learns dense word embeddings from surrounding words. They are different forms of text representation.

In [1]:
from pathlib import Path
import re
import zlib
import numpy as np
import pandas as pd
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "download_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository or one of its lab folders.")

## Task 1 - Cosine similarity

Represent the four required sentences with TF-IDF, then compare their vectors using cosine similarity. With these nonnegative vectors, scores range from 0 to 1. The default vectorizer lowercases text and ignores punctuation and word order.

In [2]:
documents = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?"
]
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(documents)
similarities = cosine_similarity(vectors)
labels = [f"Document {i}" for i in range(1, 5)]
print(pd.DataFrame({"document": labels, "text": documents}).to_string(index=False))
similarity_table = pd.DataFrame(similarities, index=labels, columns=labels)
similarity_table.round(4)

  document                                  text
Document 1           This is the first document.
Document 2 This document is the second document.
Document 3            And this is the third one.
Document 4           Is this the first document?


,Document 1,Document 2,Document 3,Document 4
Document 1,1.0000,0.6469,0.3078,1.0000
Document 2,0.6469,1.0000,0.2252,0.6469
Document 3,0.3078,0.2252,1.0000,0.3078
Document 4,1.0000,0.6469,0.3078,1.0000


### Interpretation

Documents 1 and 4 contain the same words after normalization, so their cosine similarity is **1.0**, despite their different word order and punctuation. The diagonal is also 1.0 because each nonempty vector is identical to itself. This representation measures weighted word overlap, not a full understanding of sentence meaning.

## Task 2 - Important words using TF-IDF

Use the three required sentences. Show the complete TF-IDF matrix and the five highest-weight terms in each document.

Scikit-learn's defaults use raw term counts, smoothed IDF `log((1 + N) / (1 + df)) + 1`, and L2 normalization. These differ from the simplified unsmoothed formula in the handout. Stopwords are retained to make the default behavior explicit; a high weight is statistical importance in this small corpus, not necessarily semantic importance.

In [3]:
sentences = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses",
    "data scientists analyze data"
]
tfidf = TfidfVectorizer()
matrix = tfidf.fit_transform(sentences)
terms = tfidf.get_feature_names_out()
weights = pd.DataFrame(matrix.toarray(), columns=terms,
                       index=["Sentence 1", "Sentence 2", "Sentence 3"])
weights.round(4)

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
Sentence 1,0.0000,0.0000,0.0000,0.1895,0.3209,0.3209,0.2440,0.3209,0.4881,0.2440,0.4881,0.0000,0.2440,0.0000
Sentence 2,0.0000,0.4003,0.4003,0.2364,0.0000,0.0000,0.3044,0.0000,0.3044,0.3044,0.3044,0.0000,0.3044,0.4003
Sentence 3,0.5427,0.0000,0.0000,0.6411,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5427,0.0000,0.0000


In [4]:
important_words = []
for sentence, row in weights.iterrows():
    ordered = sorted(((term, weight) for term, weight in row.items() if weight > 0),
                     key=lambda item: (-item[1], item[0]))
    for rank, (term, weight) in enumerate(ordered[:5], 1):
        important_words.append({"sentence": sentence, "rank": rank,
                                "term": term, "tfidf": weight})
pd.DataFrame(important_words).round(4)

,sentence,rank,term,tfidf
0,Sentence 1,1,of,0.4881
1,Sentence 1,2,science,0.4881
2,Sentence 1,3,fields,0.3209
3,Sentence 1,4,important,0.3209
4,Sentence 1,5,most,0.3209
5,Sentence 2,1,best,0.4003
6,Sentence 2,2,courses,0.4003
7,Sentence 2,3,this,0.4003
8,Sentence 2,4,is,0.3044
9,Sentence 2,5,of,0.3044


### Interpretation

Terms appearing in fewer documents receive higher IDF weights. Repetition also matters: `science` occurs twice in Sentence 1, and `data` occurs twice in Sentence 3. Because `data` appears in all three sentences, its IDF is lower than that of terms appearing only once in the corpus. Final weights combine these effects and are normalized within each document.

In the output, `science` and `of` tie at 0.4881 in Sentence 1; `best`, `courses`, and `this` tie at 0.4003 in Sentence 2; and `data` has the highest weight (0.6411) in Sentence 3. The high weights for `of` and `this` illustrate why optional stopword filtering may be useful when the goal is to highlight content words.

## Task 3 - Train Skip-gram Word2Vec on Simpsons dialogue

Use the full downloaded CSV and its `spoken_words` column. Exclude missing or whitespace-only spoken text. Keep repeated dialogue because its frequency contributes to the model's context statistics; no sampling is applied.

The cleaning function below follows the supplied procedure exactly. It lowercases text, removes digits, and removes the listed punctuation. Other punctuation is retained. Tokenize each spoken line separately, so context windows do not cross script-line boundaries. `preserve_line=True` uses NLTK word tokenization without requiring a sentence-tokenizer download.

In [5]:
path = ROOT / "Lab 5/data/simpsons_script_lines.csv"
if not path.is_file():
    raise FileNotFoundError("Run python download_data.py from the repository first.")
scripts = pd.read_csv(path, low_memory=False)
print("Source rows:", len(scripts))
print("Missing spoken_words:", scripts["spoken_words"].isna().sum())
spoken = scripts["spoken_words"].dropna().astype(str)
print("Whitespace-only spoken lines:", spoken.str.strip().eq("").sum())
spoken = spoken[spoken.str.strip().ne("")]
print("Nonempty spoken lines:", len(spoken))

Source rows: 158271
Missing spoken_words: 26159
Whitespace-only spoken lines: 0
Nonempty spoken lines: 132112


In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,??.?$-]", "", text)
    return text

cleaned = spoken.map(clean_text)
tokens = [nltk.word_tokenize(text, preserve_line=True) for text in cleaned]
tokens = [line for line in tokens if line]
print("Tokenized lines:", len(tokens))
print("Total tokens:", sum(map(len, tokens)))
print("Distinct tokens:", len({word for line in tokens for word in line}))

Tokenized lines: 132096
Total tokens: 1456289
Distinct tokens: 44324


### Model configuration

Use the required `min_count=1`, `vector_size=100`, `window=5`, and `sg=1` (Skip-gram). Set `epochs=5` explicitly to match the usual default. A fixed seed, one training worker, and a stable hash function reduce run-to-run variation. Minor differences can still occur across package versions and hardware.

These are exploratory word relationships within the training corpus. The assignment does not request a classifier, held-out accuracy, or a train/test split.

In [7]:
def stable_hash(word):
    return zlib.crc32(word.encode("utf-8"))

Skip_gram_model = Word2Vec(
    sentences=tokens,
    min_count=1,
    vector_size=100,
    window=5,
    sg=1,
    epochs=5,
    seed=42,
    workers=1,
    hashfxn=stable_hash
)
print("Vocabulary size:", len(Skip_gram_model.wv))
print("Vector dimensions:", Skip_gram_model.wv.vector_size)
required_words = ["homer", "marge", "bart", "jimbo", "milhouse", "kearney", "nelson", "patty", "selma"]
missing_words = [word for word in required_words if word not in Skip_gram_model.wv]
if missing_words:
    raise ValueError(f"Required query words are absent: {missing_words}")
print("All requested character names are present.")

Vocabulary size: 44324
Vector dimensions: 100
All requested character names are present.


## Task 4 - Words similar to Homer, Marge, and Bart

`wv.most_similar()` ranks vocabulary words by cosine similarity to the requested word vector. Show the top 10 for each character. These scores indicate similar usage contexts, not the probability that two characters have a particular relationship.

In [8]:
neighbors = []
for character in ["homer", "marge", "bart"]:
    for rank, (word, score) in enumerate(Skip_gram_model.wv.most_similar(character, topn=10), 1):
        neighbors.append({"query": character, "rank": rank,
                          "similar_word": word, "cosine_similarity": score})
neighbor_table = pd.DataFrame(neighbors)
print(neighbor_table.round(4).to_string(index=False))

query  rank similar_word  cosine_similarity
homer     1          abe             0.8870
homer     2        marge             0.8597
homer     3         bart             0.8460
homer     4       grampa             0.8341
homer     5        eliza             0.8298
homer     6          ned             0.8294
homer     7         herb             0.8117
homer     8  bartholomew             0.8104
homer     9       mister             0.8079
homer    10    krabappel             0.8076
marge     1          abe             0.8669
marge     2        homer             0.8597
marge     3         lisa             0.8258
marge     4        becky             0.8222
marge     5          apu             0.8206
marge     6   sweetheart             0.8143
marge     7      sweetie             0.8126
marge     8        eliza             0.8099
marge     9        honey             0.8091
marge    10      lurleen             0.8049
 bart     1         lisa             0.8568
 bart     2          abe        

## Task 5 - Find the odd word out

Pass each requested group as a list to `wv.doesnt_match()`. The method selects the word least aligned with the group's average normalized vector. The result is learned from dialogue contexts and need not match a human's interpretation of character relationships.

In [9]:
groups = [
    ["jimbo", "milhouse", "kearney"],
    ["nelson", "bart", "milhouse"],
    ["homer", "patty", "selma"]
]
odd_results = []
for group in groups:
    odd_results.append({"words": ", ".join(group),
                        "odd_one_out": Skip_gram_model.wv.doesnt_match(group)})
pd.DataFrame(odd_results)

,words,odd_one_out
0,"jimbo, milhouse, kearney",milhouse
1,"nelson, bart, milhouse",nelson
2,"homer, patty, selma",homer


## Conclusion

All five tasks are completed with the inputs specified in the handout. TF-IDF provides inspectable term weights and document overlap scores; Skip-gram learns contextual word vectors from dialogue. Nearest-neighbor and odd-one-out results are model outputs, not definitive facts about the characters. Rare tokens are retained because the assignment requires `min_count=1`, so some vectors may be weakly supported.

### Observed results

The source contains 158,271 rows. After excluding 26,159 missing spoken entries and 16 lines that produce no tokens after cleaning, the model uses **132,096 lines**, **1,456,289 tokens**, and a **44,324-token vocabulary**.

The closest words in this run are `abe` for `homer`, `abe` for `marge`, and `lisa` for `bart`. The odd-one-out answers, in task order, are **milhouse**, **nelson**, and **homer**.
